# Evaluation

Compares a corpus of VLM transcriptions against the manually annotated ground truth of the
[corpus-show-prog-avignon](https://github.com/stage-to-data/corpus-show-prog-avignon) repository.

Metrics are reported at three granularities:

1. **file level** — Levenshtein distance/ratio over the whole page;
2. **unit level** — lines/sentences are matched across the two texts, then precision/recall and
   WER/CER/Levenshtein are computed on the matched pairs;
3. **named entity level** — same, on entities extracted with spaCy.

Unit matching is order-insensitive on purpose: a VLM may return the blocks of a programme page in a
different order than the reference without that being a transcription error. Precision/recall then
measure coverage, and WER/CER/Levenshtein on matches measure fidelity.

All corpus-wide figures are weighted by page word count, so that a two-word page does not weigh as
much as an 800-word one.


## 0. Setup


In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    print('Installing dependencies...')
    %pip install -q --disable-pip-version-check -r requirements.txt
    %pip install -q --disable-pip-version-check -e ..
    !python -m spacy download xx_ent_wiki_sm

import os
import unicodedata
from pathlib import Path

import Levenshtein
import spacy

from eval_utils import (
    clean_text,
    compute_wer_cer,
    draw_scatter,
    get_mean,
    get_weighted_mean,
    match_units,
    matched_strings,
    split_text,
)
from ptod.utils import collect_files, read_txt, write_json, write_txt

print('\N{THUMBS UP SIGN} Ready!')


Pick the run to score. `GROUND_TRUTH_COMMIT` is recorded in the results so that a figure can
always be traced back to the exact state of the reference corpus it was computed against — the
ground truth is revised over time and runs made against different states are not comparable.
See `ground-truth/README.md`.


In [ ]:
# --- what to evaluate -------------------------------------------------------
CAMPAIGN  = 'final-tests'
TEST_PASS = 'prompt-4'

RUN_DIR = Path('results') / CAMPAIGN / TEST_PASS
TO_TEST_CORPUS = RUN_DIR / 'transcriptions'

# --- reference corpus ------------------------------------------------------
# Mirrored in this repository; see ground-truth/README.md for provenance and
# for the two ground truth states the published results were scored against.
EVAL_CORPUS = Path('ground-truth') / 'md'
GROUND_TRUTH_COMMIT = 'c7fc5733'  # 2025-05-05, 151 pages / 256,585 characters

# --- where to write --------------------------------------------------------
OUT_PATH = RUN_DIR

for folder in (TO_TEST_CORPUS, EVAL_CORPUS):
    if not folder.is_dir():
        raise FileNotFoundError(f'Not found: {folder.resolve()}')

eval_files = collect_files(str(EVAL_CORPUS), ['md'])
to_test_files = collect_files(str(TO_TEST_CORPUS), ['md'])

print(f'\N{LEFT-POINTING MAGNIFYING GLASS} Found {len(eval_files)} ground truth pages '
      f'and {len(to_test_files)} transcriptions to score.')


Pair each file to test with its reference by file name.

File names are compared after Unicode NFC normalisation. Accented names such as
`...MiseEnScèneDePatriceChéreau...` exist on disk in both composed (NFC) and decomposed (NFD)
form depending on the operating system and sync client that wrote them. Without normalisation the
two forms compare as different strings and the affected pages are silently dropped from the
evaluation as "missing".


In [ ]:
def key_of(path):
    """Normalised basename without extension, used to pair test and reference files."""
    return unicodedata.normalize('NFC', os.path.splitext(os.path.basename(path))[0])

eval_map = {key_of(item): item for item in eval_files}

results = {
    'global_results': {},
    'evaluation_metadata': {
        'source_folder': str(EVAL_CORPUS),
        'ground_truth_commit': GROUND_TRUTH_COMMIT,
    },
    'to_test_metadata': {'source_folder': str(TO_TEST_CORPUS),
                         'campaign': CAMPAIGN, 'test_pass': TEST_PASS},
    'files': [],
}

missing_in_eval = []
for item in to_test_files:
    key = key_of(item)
    if key in eval_map:
        results['files'].append({'to_test_file': item, 'eval_file': eval_map[key]})
    else:
        missing_in_eval.append(key)

if missing_in_eval:
    print('\N{WARNING SIGN} Missing files (present in to test but not in evaluation corpus):')
    for item in missing_in_eval:
        print(' ', item)
else:
    print('\N{THUMBS UP SIGN} No missing files!')

print(f"\nFinal test corpus is {len(results['files'])} files.")


## 1. Word count weights

Word and character counts per page, used to weight the corpus-wide means.


In [ ]:
nlp = spacy.load('xx_ent_wiki_sm')

word_total = 0
char_total = 0

for file in results['files']:
    read_in = read_txt(file['eval_file'])

    doc = nlp(read_in)
    words = [token for token in doc if token.is_alpha]
    file['word_count'] = len(words)
    word_total += len(words)

    for_chars = read_in.replace('#', '').replace('\n', '')
    file['char_count'] = len(for_chars)
    char_total += len(for_chars)

min_wc = min(f['word_count'] for f in results['files'])
max_wc = max(f['word_count'] for f in results['files'])
min_cc = min(f['char_count'] for f in results['files'])
max_cc = max(f['char_count'] for f in results['files'])

print(f'\N{LEFT-POINTING MAGNIFYING GLASS} Total words : {word_total} ({min_wc}-{max_wc})')
print(f'\N{LEFT-POINTING MAGNIFYING GLASS} Total characters : {char_total} ({min_cc}-{max_cc})')

for file in results['files']:
    file['weight_word_count'] = file['word_count'] / word_total
    file['weight_char_count'] = file['char_count'] / char_total

results['evaluation_metadata'].update({
    'word_count': word_total,
    'char_count': char_total,
    'word_count_min': min_wc,
    'word_count_max': max_wc,
    'char_count_min': min_cc,
    'char_count_max': max_cc,
})


## 2. File-level evaluation: Levenshtein


In [ ]:
for file in results['files']:
    to_test_read = read_txt(file['to_test_file'])
    eval_read = read_txt(file['eval_file'])

    to_test_parsed = clean_text(to_test_read)
    eval_parsed = clean_text(eval_read)

    file['levenshtein_distance_raw'] = Levenshtein.distance(to_test_read, eval_read)
    file['levenshtein_similarity_raw'] = Levenshtein.ratio(to_test_read, eval_read)
    file['levenshtein_distance_parsed'] = Levenshtein.distance(to_test_parsed, eval_parsed)
    file['levenshtein_similarity_parsed'] = Levenshtein.ratio(to_test_parsed, eval_parsed)

for key in ['levenshtein_distance_raw', 'levenshtein_similarity_raw',
            'levenshtein_distance_parsed', 'levenshtein_similarity_parsed']:
    results['global_results'][f'mean_{key}'] = get_mean(results, key)
    results['global_results'][f'mean_{key}_weighted_word'] = get_weighted_mean(
        results, key, 'weight_word_count')
    results['global_results'][f'mean_{key}_weighted_char'] = get_weighted_mean(
        results, key, 'weight_char_count')

g = results['global_results']
print(f"\N{BAR CHART} Mean Levenshtein ratio (raw)    : {g['mean_levenshtein_similarity_raw']:.4f}")
print(f"\N{BAR CHART} Mean Levenshtein ratio (parsed) : {g['mean_levenshtein_similarity_parsed']:.4f}")
print(f"\N{BAR CHART} Word-weighted, raw              : "
      f"{g['mean_levenshtein_similarity_raw_weighted_word']:.4f}")
print(f"\N{BAR CHART} Word-weighted, parsed           : "
      f"{g['mean_levenshtein_similarity_parsed_weighted_word']:.4f}")


## 3. Unit-level evaluation

Units are lines/sentences. Pairs are formed by greedy best match above `SIMILARITY_THRESHOLD`;
WER, CER and Levenshtein are then computed on those pairs.

`match_units` returns the pairs as an explicit list of `(predicted_index, reference_index)` tuples,
which is passed straight to `compute_wer_cer`. Do not decompose it into two separate collections:
recombining them by iteration order pairs each prediction with an arbitrary reference and the
resulting error rates are then meaningless.


In [ ]:
SIMILARITY_THRESHOLD = 0.8

for file in results['files']:
    to_test_units = split_text(read_txt(file['to_test_file']))
    eval_units = split_text(read_txt(file['eval_file']))

    precision, recall, pairs = match_units(
        to_test_units, eval_units, SIMILARITY_THRESHOLD, 'l', True)
    avg_wer, avg_cer, avg_lev = compute_wer_cer(to_test_units, eval_units, pairs)

    file['precision'] = precision
    file['recall'] = recall
    file['mean_wer_on_matches'] = avg_wer
    file['mean_cer_on_matches'] = avg_cer
    file['mean_levenshtein_on_matches'] = avg_lev

for key in ['precision', 'recall', 'mean_wer_on_matches',
            'mean_cer_on_matches', 'mean_levenshtein_on_matches']:
    results['global_results'][f'mean_{key}'] = get_mean(results, key)
    results['global_results'][f'mean_{key}_word_weighted'] = get_weighted_mean(
        results, key, 'weight_word_count')

g = results['global_results']
print(f"\N{BAR CHART} Mean precision : {g['mean_precision']:.4f} "
      f"(word-weighted {g['mean_precision_word_weighted']:.4f})")
print(f"\N{BAR CHART} Mean recall    : {g['mean_recall']:.4f} "
      f"(word-weighted {g['mean_recall_word_weighted']:.4f})")
print(f"\N{BAR CHART} Mean Levenshtein on matches : "
      f"{g['mean_levenshtein_on_matches_word_weighted']:.4f} (word-weighted)")


## 4. Named entity evaluation

Same procedure, on entities extracted with spaCy. For this project precise named entity recognition
matters more than strict section ordering, which is why these figures are reported separately.


In [ ]:
for file in results['files']:
    to_test_parsed = clean_text(read_txt(file['to_test_file']))
    eval_parsed = clean_text(read_txt(file['eval_file']))

    test_entities = [ent.text for ent in nlp(to_test_parsed).ents]
    eval_entities = [ent.text for ent in nlp(eval_parsed).ents]

    precision, recall, pairs = match_units(
        test_entities, eval_entities, SIMILARITY_THRESHOLD, 'l', True)
    avg_wer, avg_cer, avg_lev = compute_wer_cer(test_entities, eval_entities, pairs)

    file['matches'] = matched_strings(test_entities, eval_entities, pairs)
    file['ner_precision'] = precision
    file['ner_recall'] = recall
    file['mean_wer_on_matches_ner'] = avg_wer
    file['mean_cer_on_matches_ner'] = avg_cer
    file['mean_levenshtein_on_matches_ner'] = avg_lev

for key in ['ner_precision', 'ner_recall', 'mean_wer_on_matches_ner',
            'mean_cer_on_matches_ner', 'mean_levenshtein_on_matches_ner']:
    results['global_results'][f'mean_{key}'] = get_mean(results, key)
    results['global_results'][f'mean_{key}_word_weighted'] = get_weighted_mean(
        results, key, 'weight_word_count')

g = results['global_results']
print(f"\N{BAR CHART} Mean NER precision : {g['mean_ner_precision']:.4f} "
      f"(word-weighted {g['mean_ner_precision_word_weighted']:.4f})")
print(f"\N{BAR CHART} Mean NER recall    : {g['mean_ner_recall']:.4f} "
      f"(word-weighted {g['mean_ner_recall_word_weighted']:.4f})")
print(f"\N{BAR CHART} Mean Levenshtein on NER matches : "
      f"{g['mean_levenshtein_on_matches_ner_word_weighted']:.4f} (word-weighted)")


## 5. Output results


In [ ]:
OUT_PATH.mkdir(parents=True, exist_ok=True)
write_json(str(OUT_PATH / 'scores.json'), results)
print(f'\N{THUMBS UP SIGN} Saved to {OUT_PATH}')


In [ ]:
draw_scatter(results, 'precision', 'mean_cer_on_matches_ner',
             'Precision/Mean CER on NER matches',
             str(OUT_PATH / 'img_precision_cer.png'))

draw_scatter(results, 'recall', 'mean_cer_on_matches_ner',
             'Recall/Mean CER on NER matches',
             str(OUT_PATH / 'img_recall_cer.png'), 'orange')

draw_scatter(results, '&&FILE', 'precision',
             'Precision',
             str(OUT_PATH / 'img_precision.png'), 'green')

draw_scatter(results, '&&FILE', 'levenshtein_similarity_parsed',
             'Levenshtein ratio',
             str(OUT_PATH / 'img_lev.png'), 'black')


In [ ]:
GET_WORST = 10

worst_ner = sorted(results['files'],
                   key=lambda x: x['mean_cer_on_matches_ner'], reverse=True)[:GET_WORST]
worst_lev = sorted(results['files'],
                   key=lambda x: x['levenshtein_similarity_parsed'])[:GET_WORST]


A browsable markdown report of the run.


In [ ]:
g = results['global_results']
m = results['evaluation_metadata']
lines = []

lines.append(f'# Evaluation — {CAMPAIGN} / {TEST_PASS}\n')
lines.append(
    f'`{TO_TEST_CORPUS}` (**{len(to_test_files)} files**) was tested against '
    f'`{EVAL_CORPUS}` at commit `{GROUND_TRUTH_COMMIT}` (**{len(eval_files)} files**). '
    f'Found **{len(missing_in_eval)} missing files**.\n'
)
lines.append(
    f"Evaluation word count : {m['word_count']} ({m['word_count_min']}-{m['word_count_max']}), "
    f"character count: {m['char_count']} ({m['char_count_min']}-{m['char_count_max']})\n"
)

lines.append('## Full file-level analyses\n')
for label, key in [('raw', 'mean_levenshtein_similarity_raw'),
                   ('parsed', 'mean_levenshtein_similarity_parsed'),
                   ('raw, weighted by word', 'mean_levenshtein_similarity_raw_weighted_word'),
                   ('parsed, weighted by word', 'mean_levenshtein_similarity_parsed_weighted_word')]:
    lines.append(f'- **Mean Levenshtein ratio ({label})** : {g[key]:.4f}.')
lines.append('')
lines.append('![Levenshtein ratio](img_lev.png "Levenshtein ratio")\n')
lines.append(f'### {GET_WORST} worst Levenshtein ratios\n')
for item in worst_lev:
    lines.append(f"- {item['levenshtein_similarity_parsed']:.4f} : "
                 f"{os.path.basename(item['to_test_file'])}")
lines.append('')

for title, keys, img in [
    ('Precision/recall',
     ['precision', 'recall', 'mean_wer_on_matches', 'mean_cer_on_matches',
      'mean_levenshtein_on_matches'],
     'img_precision.png'),
    ('NER precision/recall',
     ['ner_precision', 'ner_recall', 'mean_wer_on_matches_ner', 'mean_cer_on_matches_ner',
      'mean_levenshtein_on_matches_ner'],
     None),
]:
    lines.append(f'## {title}\n')
    for key in keys:
        lines.append(f"- **{key}** : {g['mean_' + key]:.4f} "
                     f"(word-weighted {g['mean_' + key + '_word_weighted']:.4f}).")
    lines.append('')
    if img:
        lines.append(f'![{title}]({img} "{title}")\n')

lines.append(f'### {GET_WORST} worst NER CER scores\n')
for item in worst_ner:
    lines.append(f"#### {os.path.basename(item['to_test_file'])}\n")
    lines.append(f"NER precision **{item['ner_precision']:.4f}**, "
                 f"mean CER on matches **{item['mean_cer_on_matches_ner']:.4f}**\n")
    lines.append('|Prediction|Reference|')
    lines.append('|---|---|')
    for p, r in zip(item['matches']['matched_pred'], item['matches']['matched_ref']):
        lines.append(f'|{p}|{r}|')
    lines.append('')

write_txt(str(OUT_PATH / 'scores.md'), '\n'.join(lines))
print(f"\N{THUMBS UP SIGN} Saved report to {OUT_PATH / 'scores.md'}")


In [ ]:
print('\N{FLEXED BICEPS} Done!')
